In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

## Configuración de rutas y columnas

In [ ]:
data_folder = r"C:\Users\erik-\OneDrive\Escritorio\UPIIT\proyecto_actualizaciones"

# Archivos de datos UNSW-NB15
files = [
    "UNSW-NB15_1.csv",
    "UNSW-NB15_2.csv",
    "UNSW-NB15_3.csv",
    "UNSW-NB15_4.csv",
]

# Leer nombres de columnas desde el archivo de features
features_file = os.path.join(data_folder, "NUSW-NB15_features.csv")
FEATURE_NAMES = (
    pd.read_csv(features_file, encoding='latin-1')["Name"]
    .str.strip()
    .tolist()
)

print(f"Columnas cargadas: {len(FEATURE_NAMES)}")

## Features seleccionadas

Basadas en el análisis de correlación con `Label`.
Se descartaron variables redundantes entre sí (ej. `swin`/`dwin`, `Sjit`/`Djit`, `stcpb`/`dtcpb`).

In [ ]:
# Ordenadas de mayor a menor correlación con Label:
# sttl (0.837), ct_state_ttl (0.743), dttl (0.454),
# tcprtt (0.331), ackdat (0.329), synack (0.296),
# Sload (0.254), dmeansz (0.137), Dload (0.120),
# swin (0.111), ct_dst_sport_ltm (0.110), Sjit (0.075)

selected_features = [
    'dur',               # duración del flujo
    'sbytes',            # bytes enviados
    'dbytes',            # bytes recibidos
    'sttl',              # TTL fuente          | corr=0.837
    'dttl',              # TTL destino         | corr=0.454
    'Sload',             # bits/s fuente       | corr=0.254
    'Dload',             # bits/s destino      | corr=0.120
    'swin',              # ventana TCP fuente  | corr=0.111
    'smeansz',           # tamaño medio paquete fuente
    'dmeansz',           # tamaño medio paquete destino | corr=0.137
    'Sjit',              # jitter fuente       | corr=0.075
    'Sintpkt',           # tiempo entre paquetes fuente
    'tcprtt',            # RTT TCP             | corr=0.331
    'ackdat',            # tiempo ACK-DATA     | corr=0.329
    'ct_state_ttl',      # conteo estado+TTL   | corr=0.743
    'ct_srv_dst',        # conteo conexiones al mismo servicio/destino
    'ct_dst_sport_ltm',  # conteo conexiones al mismo dst+sport | corr=0.110
    'ct_dst_src_ltm',    # conteo conexiones entre mismo src-dst | corr=0.054
]

print(f"Features seleccionadas: {len(selected_features)}")

## Función de preprocesamiento

In [ ]:
def preprocess_unsw(file_path):

    df = pd.read_csv(
        file_path,
        header=None,
        names=FEATURE_NAMES,
        encoding='latin-1',
        low_memory=False
    )

    # ── Timestamp real (Unix epoch → datetime) ───────────────────────────
    df["time"] = pd.to_datetime(df["Stime"], unit="s")
    df = df.sort_values("time").reset_index(drop=True)
    df = df.set_index("time")

    # ── Seleccionar features + etiquetas ────────────────────────────────
    cols_needed = selected_features + ["Label", "attack_cat"]
    df = df[cols_needed].copy()

    # ── Limpiar infinitos y NaN ──────────────────────────────────────────
    df[selected_features] = (
        df[selected_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    # ── Agregación por ventana de 1 segundo (tiempo real) ───────────────
    agg_dict = {feat: "mean" for feat in selected_features}
    agg_dict["dur"]    = "mean"
    agg_dict["sbytes"] = "sum"
    agg_dict["dbytes"] = "sum"

    ts_features = df[selected_features].resample("1s").agg(agg_dict)

    # Conteo de flujos por ventana (feature adicional)
    ts_features["flow_count"] = df["Label"].resample("1s").size()

    # ── Etiqueta: 1 si algún flujo en la ventana es ataque ───────────────
    ts_label = df["Label"].resample("1s").max()   # 0 o 1

    # Categoría dominante de ataque en la ventana (para análisis)
    ts_attack_cat = df["attack_cat"].resample("1s").apply(
        lambda x: x[x.notna() & (x != "")].mode()[0]
        if not x[x.notna() & (x != "")].empty else "Normal"
    )

    ts = ts_features.copy()
    ts["Label"]      = ts_label
    ts["attack_cat"] = ts_attack_cat
    ts = ts.fillna(0)

    return ts

## Procesamiento y guardado de archivos

In [ ]:
processed_folder = os.path.join(data_folder, "processed")
os.makedirs(processed_folder, exist_ok=True)

processed = {}

for file in files:
    path = os.path.join(data_folder, file)
    print(f"Procesando: {file} ...", end=" ")

    ts = preprocess_unsw(path)

    processed[file] = ts

    out_name = file.replace(".csv", "_processed.csv")
    out_path = os.path.join(processed_folder, out_name)
    ts.to_csv(out_path)

    ataques   = (ts["Label"] == 1).sum()
    normales  = (ts["Label"] == 0).sum()
    pct       = 100 * ataques / len(ts) if len(ts) > 0 else 0

    print(f"OK → {len(ts):,} ventanas | Normal: {normales:,} | Ataque: {ataques:,} ({pct:.2f}%)")

print(f"\nTotal archivos procesados: {len(processed)}")

## Verificación del resultado

In [ ]:
# Mostrar resumen del primer archivo procesado como muestra
nombre_muestra = files[0]
ts_muestra = processed[nombre_muestra]

print(f"Archivo: {nombre_muestra}")
print(f"Shape:   {ts_muestra.shape}")
print(f"\nRango temporal:")
print(f"  Inicio: {ts_muestra.index.min()}")
print(f"  Fin:    {ts_muestra.index.max()}")
print(f"\nPrimeras filas:")
print(ts_muestra.head())
print(f"\nDistribución de Label:")
print(ts_muestra["Label"].value_counts().rename({0: "Normal", 1: "Ataque"}))
print(f"\nCategorías de ataque presentes:")
print(ts_muestra["attack_cat"].value_counts().head(10))